In [7]:
import pandas as pd
import numpy as np
import random as rnd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

# visualization
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

# machine learning
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import Perceptron
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler

# Acquire data
train_df = pd.read_csv('/train.csv')
test_df = pd.read_csv('/test.csv')
combine = [train_df, test_df]

# Wrangle data - Dropping features and feature engineering
print("Before", train_df.shape, test_df.shape, combine[0].shape, combine[1].shape)

# Create a new 'Title' feature
for dataset in combine:
    dataset['Title'] = dataset.Name.str.extract(' ([A-Za-z]+)\.', expand=False)

# Map titles to a numerical representation
title_mapping = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Rare": 5}
for dataset in combine:
    dataset['Title'] = dataset['Title'].replace(['Lady', 'Countess', 'Capt', 'Col',
                                                  'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    dataset['Title'] = dataset['Title'].replace('Mlle', 'Miss')
    dataset['Title'] = dataset['Title'].replace('Ms', 'Miss')
    dataset['Title'] = dataset['Title'].replace('Mme', 'Mrs')
    dataset['Title'] = dataset['Title'].map(title_mapping)
    dataset['Title'] = dataset['Title'].fillna(0)

# Create new 'FamilySize' and 'IsAlone' features
for dataset in combine:
    dataset['FamilySize'] = dataset['SibSp'] + dataset['Parch'] + 1
    dataset['IsAlone'] = 0
    dataset.loc[dataset['FamilySize'] == 1, 'IsAlone'] = 1

# Extract Cabin Deck from Cabin feature
for dataset in combine:
    dataset['Deck'] = dataset['Cabin'].str.extract('([A-Za-z])', expand=False)
    # Fill missing Deck values with 'M' for missing
    dataset['Deck'] = dataset['Deck'].fillna('M')
    # Map Deck to numerical values
    deck_mapping = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7, 'T': 8, 'M': 0}
    dataset['Deck'] = dataset['Deck'].map(deck_mapping).astype(int)

# Drop redundant features
train_df = train_df.drop(['Name', 'PassengerId', 'Ticket', 'Cabin', 'SibSp', 'Parch', 'FamilySize'], axis=1)
test_df = test_df.drop(['Name', 'Ticket', 'Cabin', 'SibSp', 'Parch', 'FamilySize'], axis=1)
combine = [train_df, test_df]

# Convert 'Sex' and 'Embarked' categorical features to numerical
for dataset in combine:
    dataset['Sex'] = dataset['Sex'].map({'female': 1, 'male': 0}).astype(int)
    dataset['Embarked'] = dataset['Embarked'].fillna(dataset['Embarked'].mode()[0])
    dataset['Embarked'] = dataset['Embarked'].map({'S': 0, 'C': 1, 'Q': 2}).astype(int)

# Impute missing 'Age' using IterativeImputer (more advanced)
imputer = IterativeImputer(RandomForestRegressor(), max_iter=10, random_state=0)
for dataset in combine:
    dataset['Age'] = imputer.fit_transform(dataset[['Age', 'Pclass', 'Sex', 'Fare', 'Embarked']]).round(0)
    dataset['Age'] = dataset['Age'].astype(int)
    # Correct for any negative or out-of-range imputed ages
    dataset.loc[dataset['Age'] < 0, 'Age'] = 0
    dataset.loc[dataset['Age'] > 80, 'Age'] = 80

# Impute missing 'Fare' with median
test_df['Fare'].fillna(test_df['Fare'].median(), inplace=True)

# Create 'Age*Class' artificial feature
for dataset in combine:
    dataset['Age*Class'] = dataset.Age * dataset.Pclass

# Apply Feature Scaling to continuous features
scaler = StandardScaler()
for dataset in combine:
    dataset['Fare_scaled'] = scaler.fit_transform(dataset[['Fare']])
    dataset['Age_scaled'] = scaler.fit_transform(dataset[['Age']])

# Update X_train and X_test with new features and scaled values
X_train = train_df.drop("Survived", axis=1)
Y_train = train_df["Survived"]
X_test  = test_df.drop("PassengerId", axis=1).copy()
X_train.shape, Y_train.shape, X_test.shape

# Dropping unscaled features for Logistic Regression and SVM
X_train_scaled = X_train.drop(['Fare', 'Age'], axis=1)
X_test_scaled = X_test.drop(['Fare', 'Age'], axis=1)

# Now evaluate models with the improved data processing
# Logistic Regression (using scaled data)
logreg = LogisticRegression(solver='liblinear')
logreg.fit(X_train_scaled, Y_train)
Y_pred = logreg.predict(X_test_scaled)
acc_log = round(logreg.score(X_train_scaled, Y_train) * 100, 2)
print(f"Logistic Regression Accuracy: {acc_log}")

# Support Vector Machines (using scaled data)
svc = SVC(gamma='auto')
svc.fit(X_train_scaled, Y_train)
Y_pred = svc.predict(X_test_scaled)
acc_svc = round(svc.score(X_train_scaled, Y_train) * 100, 2)
print(f"SVM Accuracy: {acc_svc}")

# KNN (using scaled data)
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train_scaled, Y_train)
Y_pred = knn.predict(X_test_scaled)
acc_knn = round(knn.score(X_train_scaled, Y_train) * 100, 2)
print(f"KNN Accuracy: {acc_knn}")

# Naive Bayes
gaussian = GaussianNB()
gaussian.fit(X_train, Y_train)
Y_pred = gaussian.predict(X_test)
acc_gaussian = round(gaussian.score(X_train, Y_train) * 100, 2)
print(f"Naive Bayes Accuracy: {acc_gaussian}")

# Decision Tree (using unscaled data)
decision_tree = DecisionTreeClassifier()
decision_tree.fit(X_train, Y_train)
Y_pred = decision_tree.predict(X_test)
acc_decision_tree = round(decision_tree.score(X_train, Y_train) * 100, 2)
print(f"Decision Tree Accuracy: {acc_decision_tree}")

# Random Forest (using unscaled data)
random_forest = RandomForestClassifier(n_estimators=100)
random_forest.fit(X_train, Y_train)
Y_pred = random_forest.predict(X_test)
acc_random_forest = round(random_forest.score(X_train, Y_train) * 100, 2)
print(f"Random Forest Accuracy: {acc_random_forest}")

# Linear SVC (using scaled data)
linear_svc = LinearSVC(dual=False, max_iter=10000)
linear_svc.fit(X_train_scaled, Y_train)
acc_linear_svc = round(linear_svc.score(X_train_scaled, Y_train) * 100, 2)
print(f"Linear SVC Accuracy: {acc_linear_svc}")

# Perceptron (using scaled data)
perceptron = Perceptron(max_iter=1000, tol=1e-3)
perceptron.fit(X_train_scaled, Y_train)
acc_perceptron = round(perceptron.score(X_train_scaled, Y_train) * 100, 2)
print(f"Perceptron Accuracy: {acc_perceptron}")

# Stochastic Gradient Descent (using scaled data)
sgd = SGDClassifier(max_iter=1000, tol=1e-3)
sgd.fit(X_train_scaled, Y_train)
acc_sgd = round(sgd.score(X_train_scaled, Y_train) * 100, 2)
print(f"Stochastic Gradient Descent Accuracy: {acc_sgd}")

# Evaluate the models
models = pd.DataFrame({
    'Model': ['Logistic Regression', 'Support Vector Machines', 'KNN',
              'Naive Bayes', 'Decision Tree', 'Random Forest',
              'Linear SVC', 'Perceptron', 'Stochastic Gradient Decent'],
    'Score': [acc_log, acc_svc, acc_knn,
              acc_gaussian, acc_decision_tree, acc_random_forest,
              acc_linear_svc, acc_perceptron, acc_sgd]})

models.sort_values(by='Score', ascending=False)
#print(models)

<>:34: SyntaxWarning: invalid escape sequence '\.'
<>:34: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipython-input-1232215934.py:34: SyntaxWarning: invalid escape sequence '\.'
  dataset['Title'] = dataset.Name.str.extract(' ([A-Za-z]+)\.', expand=False)


Before (891, 12) (418, 11) (891, 12) (418, 11)


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
/tmp/ipython-input-1232215934.py:83: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_df['Fare'].fillna(test_df['Fare'].median(), inplace=True)


Logistic Regression Accuracy: 81.82
SVM Accuracy: 87.43
KNN Accuracy: 86.64
Naive Bayes Accuracy: 76.32
Decision Tree Accuracy: 98.77
Random Forest Accuracy: 98.77
Linear SVC Accuracy: 80.02
Perceptron Accuracy: 76.77
Stochastic Gradient Descent Accuracy: 77.22


,Model,Score
4,Decision Tree,98.77
5,Random Forest,98.77
1,Support Vector Machines,87.43
2,KNN,86.64
0,Logistic Regression,81.82
6,Linear SVC,80.02
8,Stochastic Gradient Decent,77.22
7,Perceptron,76.77
3,Naive Bayes,76.32
